## 1. Setup and Configuration

In [ ]:
import os
import gc
import datetime as dt
from pathlib import Path

import pandas as pd
import numpy as np
import pandas_datareader as pdr
from tqdm import tqdm

In [ ]:
"""
StockTwits Dataset Downloader

This script systematically downloads the StockTwits dataset from the AWS S3 bucket
(s3://stocktwits-nyu) to the local directory, maintaining the folder structure.

The dataset includes:
- feature_wo_messages: Features extracted without message content
- messages: Raw message data
- msg_info: Metadata for messages
- sentiments: Sentiment analysis results
- symbols: Information about stock symbols
- symbol_sentiments: Sentiment information mapped to symbols
"""

import subprocess
import sys
import os
from pathlib import Path


def check_aws_cli():
    """Check if AWS CLI is installed."""
    # Get the Python executable path
    python_exe = sys.executable
    
    try:
        result = subprocess.run(
            [python_exe, "-m", "awscli", "--version"],
            capture_output=True,
            text=True,
            check=True
        )
        print(f"✓ AWS CLI found: {result.stdout.strip()}")
        return True
    except (subprocess.CalledProcessError, FileNotFoundError):
        print("✗ AWS CLI not found.")
        print("\nPlease install AWS CLI:")
        print("  - Download from: https://aws.amazon.com/cli/")
        print("  - Or use: pip install awscli")
        return False


def list_s3_contents(s3_path):
    """List contents of an S3 path."""
    python_exe = sys.executable
    try:
        print(f"\nListing contents of: {s3_path}")
        result = subprocess.run(
            [python_exe, "-m", "awscli", "s3", "ls", "--no-sign-request", s3_path],
            capture_output=True,
            text=True,
            check=True
        )
        print(result.stdout)
        return True
    except subprocess.CalledProcessError as e:
        print(f"Error listing S3 contents: {e.stderr}")
        return False


def download_dataset(base_path=".", create_subdirs=True):
    """
    Download the entire StockTwits dataset from S3.
    
    Args:
        base_path: Local directory where data will be downloaded
        create_subdirs: If True, creates dataset/v1/data/csv structure
    """
    # S3 bucket configuration
    BASE_URL = "s3://stocktwits-nyu"
    CSV_URL = f"{BASE_URL}/dataset/v1/data/csv"
    
    # Prepare local directory
    if create_subdirs:
        local_path = Path(base_path) / "dataset" / "v1" / "data" / "csv"
    else:
        local_path = Path(base_path)
    
    local_path.mkdir(parents=True, exist_ok=True)
    print(f"\n{'='*60}")
    print(f"StockTwits Dataset Downloader")
    print(f"{'='*60}")
    print(f"Source: {CSV_URL}")
    print(f"Destination: {local_path.absolute()}")
    print(f"{'='*60}\n")
    
    # Check AWS CLI availability
    if not check_aws_cli():
        return False
    
    # List available data folders
    print("\n" + "="*60)
    print("Available data categories:")
    print("="*60)
    list_s3_contents(f"{CSV_URL}/")
    
    # Data categories to download
    categories = [
        "feature_wo_messages",
        "messages",
        "msg_info",
        "sentiments",
        "symbols",
        "symbol_sentiments"
    ]
    
    # Download each category
    print("\n" + "="*60)
    print("Starting download...")
    print("="*60 + "\n")
    
    for category in categories:
        s3_category_path = f"{CSV_URL}/{category}/"
        local_category_path = local_path / category
        
        print(f"\n{'─'*60}")
        print(f"Downloading: {category}")
        print(f"{'─'*60}")
        
        try:
            # Use aws s3 sync to download all files in the category
            python_exe = sys.executable
            result = subprocess.run(
                [
                    python_exe, "-m", "awscli",
                    "s3", "sync",
                    "--no-sign-request",
                    s3_category_path,
                    str(local_category_path)
                ],
                capture_output=True,
                text=True,
                check=True
            )
            
            # Print download progress
            if result.stdout:
                print(result.stdout)
            if result.stderr:
                print(result.stderr)
            
            # Count downloaded files
            if local_category_path.exists():
                file_count = len(list(local_category_path.glob("*.csv")))
                print(f"✓ {category}: {file_count} files downloaded")
            
        except subprocess.CalledProcessError as e:
            print(f"✗ Error downloading {category}: {e.stderr}")
            continue
    
    print("\n" + "="*60)
    print("Download Summary")
    print("="*60)
    
    # Print summary of downloaded files
    total_files = 0
    total_size = 0
    
    for category in categories:
        local_category_path = local_path / category
        if local_category_path.exists():
            files = list(local_category_path.glob("*.csv"))
            category_size = sum(f.stat().st_size for f in files)
            total_files += len(files)
            total_size += category_size
            
            size_mb = category_size / (1024 * 1024)
            print(f"{category:25} {len(files):4} files  {size_mb:8.2f} MB")
    
    print("─"*60)
    total_size_gb = total_size / (1024 * 1024 * 1024)
    print(f"{'TOTAL':25} {total_files:4} files  {total_size_gb:8.2f} GB")
    print("="*60)
    print(f"\n✓ Download complete! Data saved to: {local_path.absolute()}")
    
    return True


def download_single_category(category, base_path="."):
    """
    Download a single category of data.
    
    Args:
        category: One of the data categories (e.g., 'messages', 'sentiments')
        base_path: Local directory where data will be downloaded
    """
    BASE_URL = "s3://stocktwits-nyu"
    CSV_URL = f"{BASE_URL}/dataset/v1/data/csv"
    
    local_path = Path(base_path) / "dataset" / "v1" / "data" / "csv" / category
    local_path.mkdir(parents=True, exist_ok=True)
    
    print(f"\nDownloading {category} to {local_path.absolute()}...")
    
    if not check_aws_cli():
        return False
    
    s3_category_path = f"{CSV_URL}/{category}/"
    
    python_exe = sys.executable
    try:
        result = subprocess.run(
            [
                python_exe, "-m", "awscli",
                "s3", "sync",
                "--no-sign-request",
                s3_category_path,
                str(local_path)
            ],
            capture_output=True,
            text=True,
            check=True
        )
        
        print(result.stdout)
        if result.stderr:
            print(result.stderr)
        
        file_count = len(list(local_path.glob("*.csv")))
        print(f"✓ Downloaded {file_count} files")
        return True
        
    except subprocess.CalledProcessError as e:
        print(f"✗ Error: {e.stderr}")
        return False


if __name__ == "__main__":
    # You can modify these parameters:
    # - base_path: Where to save the data (default: current directory)
    # - create_subdirs: Whether to create dataset/v1/data/csv structure (default: True)
    
    # Download all data
    success = download_dataset(base_path=r"E:\Research_data\Stocktwits", create_subdirs=True)

    
    # Alternative: Download only specific categories
    # Uncomment the following to download only specific categories:
    # download_single_category("messages", base_path=".")
    # download_single_category("sentiments", base_path=".")
    
    if success:
        print("\n✓ All downloads completed successfully!")
    else:
        print("\n✗ Download encountered errors. Please check the output above.")
        sys.exit(1)

In [ ]:
# =============================================================================
# DIRECTORY PATHS
# =============================================================================
CODE_DIR = Path(r"C:\Users\willi\.vscode\Github\ml-from-crowd")
DATA_DIR = Path(r"E:\Research_data\Stocktwits\dataset\v1\data\csv")
FIGURES_DIR = Path(r"C:\Users\willi\.vscode\Github\Figures")

# =============================================================================
# INPUT/OUTPUT FOLDERS
# =============================================================================
INPUT_FOLDER = DATA_DIR / "feature_wo_messages"
OUTPUT_FOLDER = DATA_DIR / "feature_wo_messages_cleaned_mlcrowd"
OUTPUT_BY_YEAR_FOLDER = DATA_DIR / "cleaned_by_year_mlcrowd"

# Create output folders if they don't exist
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_BY_YEAR_FOLDER.mkdir(parents=True, exist_ok=True)

# =============================================================================
# PARAMETERS
# =============================================================================
# Date range for analysis
START_DATE = dt.date(2008, 1, 1)
END_DATE = dt.date(2024, 12, 31)

# Market hours (US/Eastern)
MARKET_OPEN = dt.time(9, 30, 0)
MARKET_CLOSE = dt.time(16, 0, 0)
PRE_MARKET_START = dt.time(4, 0, 0)
POST_MARKET_END = dt.time(20, 0, 0)

print(f"Input folder: {INPUT_FOLDER}")
print(f"Output folder: {OUTPUT_FOLDER}")
print(f"Output by year folder: {OUTPUT_BY_YEAR_FOLDER}")

## 2. Inspect Raw Data Structure

In [ ]:
# Get list of all CSV files
csv_files = sorted([f for f in os.listdir(INPUT_FOLDER) if f.endswith('.csv')])
print(f"Found {len(csv_files)} CSV files to process")

# Load first file to inspect structure
sample_file = INPUT_FOLDER / csv_files[0]
df_sample = pd.read_csv(sample_file, nrows=1000)

print(f"\nSample file: {csv_files[0]}")
print(f"Shape: {df_sample.shape}")
print(f"\nColumns: {list(df_sample.columns)}")
print(f"\nData types:\n{df_sample.dtypes}")
print(f"\nFirst few rows:")
df_sample.head()

In [ ]:
# Check sentiment distribution
print("Sentiment value counts:")
print(df_sample['sentiment'].value_counts(dropna=False))

# Check for missing values
print(f"\nMissing values:\n{df_sample.isna().sum()}")

## 3. Build Trading Days Calendar

We use Fama-French data to identify trading days. Each message is mapped to its "first close after tweet" - the first market close following the message timestamp.

In [ ]:
def build_trading_calendar(start_date, end_date):
    """
    Build a trading calendar using Fama-French data.
    Maps each calendar day to its first and second trading close.
    
    Parameters:
    -----------
    start_date : datetime.date
        Start of the date range
    end_date : datetime.date
        End of the date range
        
    Returns:
    --------
    pd.DataFrame with columns: date, business_day, first_close, second_close
    """
    print("Fetching Fama-French trading days...")
    
    # Download Fama-French data to get trading days
    ff = pdr.DataReader(
        'F-F_Research_Data_5_Factors_2x3_daily', 
        'famafrench',
        start=start_date,
        end=end_date
    )[0].reset_index()
    
    ff = ff.rename(columns={"Date": "first_close"})
    ff["second_close"] = ff["first_close"].shift(-1)
    
    # Create full calendar
    all_days = pd.date_range(start=start_date, end=end_date)
    trading_days = pd.DataFrame({"date": all_days})
    
    # Merge to identify trading days
    trading_days = pd.merge(
        trading_days, 
        ff[["first_close", "second_close"]], 
        left_on="date", 
        right_on="first_close", 
        how="left",
        indicator=True
    )
    
    trading_days["business_day"] = trading_days["_merge"] == "both"
    trading_days = trading_days[["date", "business_day", "first_close", "second_close"]]
    
    # Forward fill non-trading days to point to next trading day
    trading_days = trading_days.bfill()
    
    print(f"Trading calendar built: {trading_days['business_day'].sum()} trading days out of {len(trading_days)} total days")
    
    return trading_days

# Build the calendar
trading_calendar = build_trading_calendar(START_DATE, END_DATE)
trading_calendar.head(10)

## 4. Define Cleaning Functions

In [ ]:
def parse_timestamp(df):
    """
    Parse created_at timestamp and convert to US/Eastern timezone.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with 'created_at' column
        
    Returns:
    --------
    pd.DataFrame with parsed timestamps
    """
    df = df.copy()
    
    # Parse timestamp
    df['created_at'] = pd.to_datetime(df['created_at'])
    
    # Convert to US/Eastern timezone (handle both timezone-aware and naive)
    try:
        df['created_at'] = df['created_at'].dt.tz_convert('US/Eastern').dt.tz_localize(None)
    except TypeError:
        # If timestamps are timezone-naive, assume UTC and convert
        df['created_at'] = df['created_at'].dt.tz_localize('UTC').dt.tz_convert('US/Eastern').dt.tz_localize(None)
    
    return df


def add_temporal_features(df):
    """
    Add temporal features needed for feature extraction.
    
    Features added:
    - calendar_date: Date portion of created_at
    - time: Time portion of created_at
    - hour: Hour of day (0-23)
    - is_after_hours: Boolean indicating if message is after market close
    - session: Market session category (pre_market, market_open, etc.)
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with parsed 'created_at' column
        
    Returns:
    --------
    pd.DataFrame with temporal features added
    """
    df = df.copy()
    
    # Extract date and time components
    df['calendar_date'] = pd.to_datetime(df['created_at'].dt.date)
    df['time'] = df['created_at'].dt.time
    df['hour'] = df['created_at'].dt.hour
    
    # After hours indicator (after 4 PM)
    df['is_after_hours'] = df['time'] > MARKET_CLOSE
    
    # Market session classification (for intraday features)
    def classify_session(t):
        h = t.hour
        m = t.minute
        if m < 30:
            m = 0
        else:
            m = 30
        return f"{h:02d}{m:02d}"
        
    df['session'] = df['time'].apply(classify_session)
    
    return df


def assign_trading_date(df, trading_calendar):
    """
    Assign each message to its first market close date.
    
    Logic:
    - Messages during market hours -> same day close
    - Messages after market close on trading days -> next trading day close
    - Messages on non-trading days -> next trading day close
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with 'calendar_date' and 'is_after_hours' columns
    trading_calendar : pd.DataFrame
        Trading calendar from build_trading_calendar()
        
    Returns:
    --------
    pd.DataFrame with 'date' column (trading date)
    """
    df = df.copy()
    
    # Merge with trading calendar
    df = pd.merge(
        df, 
        trading_calendar, 
        left_on='calendar_date', 
        right_on='date', 
        how='left',
        suffixes=('', '_cal')
    )
    
    # Assign trading date based on timing
    # Default: first close after the calendar date
    df['date'] = df['first_close']
    
    # If it's a trading day AND after market close -> use second close (next trading day)
    after_hours_on_trading_day = df['business_day'] & df['is_after_hours']
    df.loc[after_hours_on_trading_day, 'date'] = df.loc[after_hours_on_trading_day, 'second_close']
    
    # Clean up temporary columns
    cols_to_drop = ['calendar_date', 'first_close', 'second_close', 'date_cal']
    cols_to_drop = [c for c in cols_to_drop if c in df.columns]
    df = df.drop(columns=cols_to_drop)
    
    return df


def clean_dataframe(df, trading_calendar):
    """
    Full cleaning pipeline for a single DataFrame.
    
    Steps:
    1. Filter for valid sentiment (Bullish/Bearish)
    2. Parse timestamps to US/Eastern
    3. Add temporal features
    4. Assign trading date
    5. Drop unnecessary columns
    
    Parameters:
    -----------
    df : pd.DataFrame
        Raw StockTwits data
    trading_calendar : pd.DataFrame
        Trading calendar from build_trading_calendar()
        
    Returns:
    --------
    pd.DataFrame: Cleaned data
    """
    # Step 1: Filter for valid sentiment
    df_cleaned = df[df['sentiment'].isin(['Bullish', 'Bearish'])].copy()
    
    if len(df_cleaned) == 0:
        return df_cleaned
    
    # Step 2: Parse timestamps
    df_cleaned = parse_timestamp(df_cleaned)
    
    # Step 3: Add temporal features
    df_cleaned = add_temporal_features(df_cleaned)
    
    # Step 4: Assign trading date
    df_cleaned = assign_trading_date(df_cleaned, trading_calendar)

    # Step 5: Create weekend and holiday indicators
    df_cleaned['is_weekend'] = df_cleaned['created_at'].dt.weekday >= 5
    df_cleaned['is_holiday'] = (df_cleaned['business_day'] == False) & (df_cleaned['is_weekend'] == False)
    
    # Step 6: Drop columns not needed for feature extraction
    cols_to_drop = ['parent_message_id', 'in_reply_to_message_id']
    cols_to_drop = [c for c in cols_to_drop if c in df_cleaned.columns]
    df_cleaned = df_cleaned.drop(columns=cols_to_drop)
    
    # Filter out rows with invalid dates
    df_cleaned = df_cleaned[df_cleaned['date'].notna()]
    
    return df_cleaned

## 5. Test Cleaning on Sample Data

In [ ]:
# Test the cleaning pipeline on sample data
df_test = pd.read_csv(sample_file, nrows=10000)
df_cleaned_test = clean_dataframe(df_test, trading_calendar)

print(f"Original rows: {len(df_test):,}")
print(f"Cleaned rows: {len(df_cleaned_test):,}")
print(f"Retention rate: {len(df_cleaned_test)/len(df_test)*100:.1f}%")

print(f"\nCleaned columns: {list(df_cleaned_test.columns)}")
print(f"\nSession distribution:")
print(df_cleaned_test['session'].value_counts())

df_cleaned_test.head()

## 6. Process All Files

In [ ]:
# Process all files and save cleaned versions
total_original_rows = 0
total_cleaned_rows = 0
processed_files = 0
failed_files = []

print("Processing all files...\n")

for csv_file in tqdm(csv_files):
    try:
        # Read the file
        input_path = INPUT_FOLDER / csv_file
        df = pd.read_csv(input_path)
        
        # Clean the dataframe
        df_cleaned = clean_dataframe(df, trading_calendar)
        
        # Save to output folder
        output_path = OUTPUT_FOLDER / csv_file
        df_cleaned.to_csv(output_path, index=False)
        
        # Update statistics
        total_original_rows += len(df)
        total_cleaned_rows += len(df_cleaned)
        processed_files += 1
        
        # Free memory
        del df, df_cleaned
        
    except Exception as e:
        failed_files.append((csv_file, str(e)))
        print(f"\nError processing {csv_file}: {e}")

gc.collect()

print(f"\n{'='*60}")
print(f"Processing complete!")
print(f"{'='*60}")
print(f"Files processed: {processed_files}/{len(csv_files)}")
print(f"Failed files: {len(failed_files)}")
print(f"Total original rows: {total_original_rows:,}")
print(f"Total cleaned rows: {total_cleaned_rows:,}")
if total_original_rows > 0:
    print(f"Retention rate: {total_cleaned_rows/total_original_rows*100:.2f}%")
print(f"\nOutput folder: {OUTPUT_FOLDER}")

## 7. Estimate Memory Requirements for Full Load

In [ ]:
# Get cleaned files
cleaned_files = sorted([f for f in os.listdir(OUTPUT_FOLDER) if f.endswith('.csv')])

# Sample files to estimate memory
SAMPLE_SIZE = min(5, len(cleaned_files))
sample_files = cleaned_files[:SAMPLE_SIZE]

print(f"Sampling {SAMPLE_SIZE} files to estimate memory usage...\n")

total_sample_rows = 0
total_sample_memory = 0

for csv_file in sample_files:
    file_path = OUTPUT_FOLDER / csv_file
    df_sample = pd.read_csv(file_path)
    
    memory_usage = df_sample.memory_usage(deep=True).sum()
    total_sample_rows += len(df_sample)
    total_sample_memory += memory_usage
    
    print(f"{csv_file}: {len(df_sample):,} rows, {memory_usage / (1024**2):.2f} MB")

# Calculate estimates
if total_sample_rows > 0:
    avg_memory_per_row = total_sample_memory / total_sample_rows
    estimated_memory = total_cleaned_rows * avg_memory_per_row
    
    print(f"\n{'='*60}")
    print(f"Memory Estimation:")
    print(f"{'='*60}")
    print(f"Average memory per row: {avg_memory_per_row:.2f} bytes")
    print(f"Total rows (cleaned): {total_cleaned_rows:,}")
    print(f"Estimated memory needed: {estimated_memory / (1024**3):.2f} GB")
    print(f"\nWith 50% overhead: {estimated_memory * 1.5 / (1024**3):.2f} GB")
    print(f"With 100% overhead: {estimated_memory * 2 / (1024**3):.2f} GB")

## 8. Merge All Cleaned Files and Save by Year

This section loads all cleaned files, merges them, and saves by year for efficient access during feature extraction.

In [ ]:
# Load all cleaned files
cleaned_files = sorted([f for f in os.listdir(OUTPUT_FOLDER) if f.endswith('.csv')])

print(f"Loading {len(cleaned_files)} cleaned files...")
print("This may take several minutes for large datasets...\n")

dfs = []
for i, csv_file in enumerate(tqdm(cleaned_files)):
    file_path = OUTPUT_FOLDER / csv_file
    df_temp = pd.read_csv(file_path)
    dfs.append(df_temp)
    
    # Progress update every 50 files
    if (i + 1) % 50 == 0:
        current_rows = sum(len(df) for df in dfs)
        print(f"  Loaded {i+1}/{len(cleaned_files)} files - {current_rows:,} rows")

print("\nConcatenating all dataframes...")
df_merged = pd.concat(dfs, ignore_index=True)

# Free memory
del dfs
gc.collect()

print(f"\n{'='*60}")
print(f"Merge Complete!")
print(f"{'='*60}")
print(f"Total rows: {len(df_merged):,}")
print(f"Total columns: {len(df_merged.columns)}")
print(f"Memory usage: {df_merged.memory_usage(deep=True).sum() / (1024**3):.2f} GB")
print(f"\nColumns: {list(df_merged.columns)}")
print(f"\nDate range: {df_merged['date'].min()} to {df_merged['date'].max()}")

In [ ]:
# Remove duplicates based on message_id
original_len = len(df_merged)
df_merged = df_merged.drop_duplicates(subset=['message_id'], keep='first')
duplicates_removed = original_len - len(df_merged)

print(f"Duplicates removed: {duplicates_removed:,} ({duplicates_removed/original_len*100:.2f}%)")
print(f"Final row count: {len(df_merged):,}")

In [ ]:
# Convert date column to datetime and extract year
df_merged['date'] = pd.to_datetime(df_merged['date'])
df_merged['year'] = df_merged['date'].dt.year.astype(int)

# Get unique years
years = sorted(df_merged['year'].unique())
print(f"Years in data: {years}")
print(f"\nRows per year:")
print(df_merged['year'].value_counts().sort_index())

In [ ]:
# Save by year
print(f"\nSaving {len(years)} yearly files to: {OUTPUT_BY_YEAR_FOLDER}\n")

rows_written = 0
for year in years:
    df_year = df_merged[df_merged['year'] == year].drop(columns=['year'])
    output_path = OUTPUT_BY_YEAR_FOLDER / f"stocktwits_cleaned_{year}.csv"
    df_year.to_csv(output_path, index=False)
    rows_written += len(df_year)
    print(f"  {year}: {len(df_year):,} rows -> {output_path.name}")

print(f"\n{'='*60}")
print(f"Done! Total rows written: {rows_written:,}")
print(f"Output folder: {OUTPUT_BY_YEAR_FOLDER}")

## 9. Data Quality Summary

In [ ]:
# Summary statistics for cleaned data
print("=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)

print(f"\n1. OVERALL STATISTICS")
print(f"   Total messages: {len(df_merged):,}")
# print(f"   Unique symbols: {df_merged['symbol'].nunique():,}")
print(f"   Unique users: {df_merged['user_id'].nunique():,}")
print(f"   Date range: {df_merged['date'].min().date()} to {df_merged['date'].max().date()}")

print(f"\n2. SENTIMENT DISTRIBUTION")
sentiment_counts = df_merged['sentiment'].value_counts()
for sentiment, count in sentiment_counts.items():
    print(f"   {sentiment}: {count:,} ({count/len(df_merged)*100:.1f}%)")

print(f"\n3. SESSION DISTRIBUTION")
session_counts = df_merged['session'].value_counts()
for session, count in session_counts.items():
    print(f"   {session}: {count:,} ({count/len(df_merged)*100:.1f}%)")

# print(f"\n4. TOP 10 SYMBOLS BY VOLUME")
# top_symbols = df_merged['symbol'].value_counts().head(10)
# for symbol, count in top_symbols.items():
#     print(f"   {symbol}: {count:,}")

print(f"\n5. MISSING VALUES")
missing = df_merged.isna().sum()
for col, count in missing.items():
    if count > 0:
        print(f"   {col}: {count:,} ({count/len(df_merged)*100:.2f}%)")
if missing.sum() == 0:
    print("   No missing values in cleaned data")

print(f"\n" + "=" * 60)
print("Cleaning complete! Data is ready for feature extraction.")
print("=" * 60)